# Time series of the yearly responses, 2001–2014, per contrast

Yearly mean differences of the variables of the BVOC/SOA/cloud context maps (`forcing_maps.CONTEXT_PANELS`) and of the NET Ghan (2013) terms, one figure per contrast (§1–2) in its category folder `results/figures/<category>/timeseries_<key>_2001_2014.png`; absolute values of every nudged run in §3 (`results/figures/timeseries/`).

**Contrasts (user 2026-11-08), one folder per category in `results/figures/<category>/`:**

| Category | Contrasts (case1 − case2) |
|---|---|
| interactive | LCC_FUT − CTRL_PD · [LCC − CTRL]_PD · [LCC − CTRL]_FUT · CTRL_[FUT − PD] · LCC_[FUT − PD] |
| fBVOC | LCC_PD_fBVOC [lccPD − ctrlPD] · LCC_FUT_fBVOC [lccFUT − ctrlFUT] · [lccFUT − lccPD] · [lccFUT − ctrlPD] · [ctrlFUT − ctrlPD] · [lccPD − ctrlPD] (not really physical) |
| non-BVOC | [LCC_PD − CTRL_PD]_fBVOCctrlPD · [LCC_FUT − CTRL_FUT]_fBVOCctrlFUT · [LCC_FUT − LCC_PD]_fBVOCctrlPD · [LCC_FUT − CTRL_PD]_fBVOCctrlPD |
| artefacts | art_LCC_PD = LCC_PD − LCC_PD_fBVOCctrlPD · art_LCC_FUT = LCC_FUT − LCC_FUT_fBVOCctrlFUT · art_CTRL_PD = CTRL_PD − CTRL_PD_fBVOCctrlPD · art_CTRL_FUT = CTRL_FUT − CTRL_FUT_fBVOCctrlFUT |

Definitions: `forcing_maps.CONTRASTS`. **`LCC_FUT_fBVOClccFUT` is a design-B stand-in** until the run (24g) is postprocessed: LCC_FUT − art_CTRL_FUT = LCC_FUT − CTRL_FUT + CTRL_FUT_fBVOCctrlFUT (assumes art(LCC land) = art(CTRL land)); rows using it say "(design B)". **To be replaced:** set `fm.LCC_FUT_FBVOCLCCFUT_DESIGN_B = False` and rerun.

**Two lines per panel**
- **global**: area-weighted global mean (land-only fields such as ET: global land mean, weights = cell area × land fraction);
- **regional**: **modified cells** for the BVOC emissions (ISOP, MTERP), **≥45°N** for all other variables.

Dashed horizontal line = mean over the years of that line; legend = mean ± 95 % CI (t-interval over years).

**Data and caveats**
- Input: `data/processed-data/postprocessing/<alias>_<atm|lnd>_<CAT>_2001-2014.nc`, CO₂-fix (new) chain.
- **Year 2000 is not shown:** first 12 months of every nudged run, dropped as spin-up in postprocessing.
- In-cloud fields (ACTNL, ACTREL) use an FCTL ≥ 0.01 mask that differs between runs, so a difference of in-cloud means is not exactly the mean of the cell differences.
- Reorganised 2026-11-08 (user); earlier figures in `results/figures-pre-reorg-20261108/timeseries/`, tables in `results/tables-pre-reorg-20261108/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))   # repo src/, no pip install needed

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from boreal_forest_expansion.plotting.style import set_default_style
from boreal_forest_expansion.plotting import forcing_maps as fm
from boreal_forest_expansion.plotting import palette as pal

set_default_style()
START_YEAR, END_YEAR = 2001, 2014
FIGDIR = fm.FIGURE_DIR / "timeseries"   # absolute values of the runs (§3); contrast figures go to the category folders
TABLE = fm.TABLE_DIR / f"timeseries_regional_means_{START_YEAR}_{END_YEAR}.csv"
FIGDIR.mkdir(parents=True, exist_ok=True)

# contrast key -> Contrast (4 categories, user 2026-11-08); design-B stand-in for LCC_FUT_fBVOClccFUT (to be replaced)
CONTRASTS = {c.key: c for rows in fm.CONTRASTS.values() for c in rows}
REGIONAL = {"sfisop": "modified cells", "sfmterp": "modified cells"}   # all other panels: ≥45°N
COLORS = {"global": pal.NAVY, "regional": pal.RED}

# panels: the 9 context maps + the 4 NET Ghan terms
PANELS = [(p.key, p.title, p.variable_class, p.variable, p.units, p.component) for p in fm.CONTEXT_PANELS]
PANELS += [(f"NET_{t}", fm.GHAN_TITLES[t], "RADIATIVE", f"NET_{t}", "W m$^{-2}$", "atm") for t in fm.GHAN_TERMS]

# every run used must exist for 2001-2014
for c1, c2 in (c.pair for c in CONTRASTS.values()):
    for alias in {**fm.run_weights(c1), **fm.run_weights(c2)}:
        for cls, comp in {(p[2], p[5]) for p in PANELS}:
            path = fm.filename(alias, cls, START_YEAR, END_YEAR, comp)
            if not path.exists():
                print("MISSING", path.name)
print("runs checked;", len(CONTRASTS), "contrasts,", len(PANELS), "panels")


## 1. Yearly global and regional means

Region weights come from a reference file of the same component (CAM and CLM grids differ by ~1e-13 in the coordinates): `fm.region_masks`, `fm.regional`.

In [ ]:
refs = {comp: fm.open_class("CTRL_PD", cls, START_YEAR, END_YEAR, comp)
        for comp, cls in (("atm", "RADIATIVE"), ("lnd", "ET"))}
masks = {comp: fm.region_masks(ds) for comp, ds in refs.items()}
for comp, m in masks.items():
    m["global"] = m["≥45°N"] | ~m["≥45°N"]   # all cells

rows = []
for label, (c1, c2) in ((k, c.pair) for k, c in CONTRASTS.items()):
    for key, title, cls, var, units, comp in PANELS:
        d = fm.yearly_difference(c1, c2, cls, var, START_YEAR, END_YEAR, comp)
        for line, region in (("global", "global"), ("regional", REGIONAL.get(key, "≥45°N"))):
            r = fm.regional(d, refs[comp], masks[comp][region])
            rows += [dict(category=CONTRASTS[label].category, contrast=label, name=CONTRASTS[label].one_line, case1=str(c1), case2=str(c2), panel=key, variable=var, line=line,
                          region=region, units=units, year=int(y), value=float(v))
                     for y, v in zip(r["year"].values, r.values)]
t = pd.DataFrame(rows)

# Ghan closure on the regional means: ARI + cloud + rest = total
g = t[t.panel.str.startswith("NET_")].pivot_table(index=["contrast", "line", "year"], columns="variable", values="value")
err = (g["NET_dir"] + g["NET_cloud"] + g["NET_rest"] - g["NET_toa"]).abs().max()
print(f"Ghan closure (regional means): max |ARI + cloud + rest - total| = {err:.1e} W m-2")
assert err < 1e-3
assert t.groupby(["contrast", "panel", "line"]).year.count().eq(END_YEAR - START_YEAR + 1).all()

t.to_csv(TABLE, index=False)
print("Saved", TABLE)


In [ ]:
def mean_ci(v, conf=0.95):
    v = np.asarray(v)
    return v.mean(), stats.t.ppf(0.5 + conf / 2, v.size - 1) * v.std(ddof=1) / np.sqrt(v.size)

# mean ± 95 % CI over years; `region` = region of the regional line
s = (t.groupby(["contrast", "panel", "units", "line"], sort=False)["value"]
       .apply(lambda v: "{:+.3g} ± {:.2g}".format(*mean_ci(v))).unstack("line"))
s["region"] = t[t.line == "regional"].groupby(["contrast", "panel", "units"], sort=False).region.first()
with pd.option_context("display.max_rows", 200):
    display(s)


## 2. Figures: one per contrast, in its category folder

Rows 1–3: context variables (same order as the context maps); row 4: NET Ghan terms (W m⁻²).

In [ ]:
def plot_contrast(label, save=True):
    c = CONTRASTS[label]
    sub = t[t.contrast == label]
    fig = plt.figure(figsize=(13, 13.5), constrained_layout=True)
    top, bottom = fig.subfigures(2, 1, height_ratios=[3, 1.05], hspace=0.04)
    axes = list(top.subplots(3, 3).ravel()) + list(bottom.subplots(1, 4))
    bottom.suptitle("Ghan decomposition, NET (positive = warming)", fontsize=11, fontweight="semibold")
    for ax, (key, title, cls, var, units, comp) in zip(axes, PANELS):
        for line in ("global", "regional"):
            d = sub[(sub.panel == key) & (sub.line == line)].sort_values("year")
            m, h = mean_ci(d.value)
            ax.plot(d.year, d.value, "o-", ms=3.5, lw=1.4, color=COLORS[line],
                    label=f"{'global' if line == 'global' else d.region.iloc[0]}: {m:+.3g} ± {h:.2g}")
            ax.axhline(m, color=COLORS[line], lw=0.9, ls="--", alpha=0.8)
        ax.axhline(0, color="black", lw=0.6)
        ax.set_title(title, fontsize=9.5, fontweight="semibold")
        ax.set_ylabel(f"Δ [{units}]", fontsize=8)
        ax.set_xticks(range(START_YEAR, END_YEAR + 1, 2))
        ax.tick_params(labelsize=7.5)
        ax.legend(fontsize=7, loc="best", frameon=False)
        ax.grid(alpha=0.25, lw=0.5)
    fig.suptitle(f"Yearly responses ({c.category}): {c.one_line} = {c.case1} − {c.case2} ({START_YEAR}–{END_YEAR})\n"
                 "global mean and modified cells (BVOC emissions) / ≥45°N (other variables); legend = mean ± 95 % CI over years",
                 fontsize=12, fontweight="bold")
    if save:
        path = fm.FIG_CATEGORY[c.category] / f"timeseries_{label}_{START_YEAR}_{END_YEAR}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        print("Saved", path)
    return fig

for label in CONTRASTS:
    fig = plot_contrast(label)
    plt.show(); plt.close(fig)


## 3. Absolute values of every nudged run (no differences)

Same 13 panels, same regions (modified cells for ISOP/MTERP, ≥45°N for the rest), yearly means of each run itself.
**4 figures:** {PD, FUT} × {global, regional}; one line per run. PD and FUT are separate because their levels differ far more than the runs within a period.

- **Colours:** CTRL navy, LCC red; fBVOC runs on LCC land: plum = CTRL_PD emissions, orange = CTRL_FUT emissions, teal = LCC_PD emissions; **dashed navy** = CTRL land with its own emissions prescribed (artefact runs).
- **Ghan panels, absolute:** F_TOTAL = net TOA flux (down positive); F_AEROSOL-DIRECT = F − F_clean (aerosol direct effect); F_CLOUD = F_clean − F_clean,clear (cloud radiative effect without aerosol); F_REST = F_clean,clear (clear-sky, aerosol-free net flux). F_TOTAL = sum of the three. Global annual F_TOTAL is not 0 in these nudged, fixed-SST runs.
- `LCC_PD_fBVOClccPD` is the 3-yr test (2000–2002), so only 2001–2002 remain after the spin-up year.
- Absolute means of in-cloud quantities (droplet radius, number) are yearly ratios of means with FCTL ≥ 0.01, as in the maps; global CLM fields (ET) = global land mean.
- **In-cloud caveat:** cells with FCTL < 0.01 are masked in each run separately, so for droplet radius and number the difference of these absolute means ≠ the §1 mean difference (cells valid in both runs). Checked below: all other fields agree to < 1e-6; for global droplet radius the gap (≈0.01 µm) exceeds the typical LCC_PD − CTRL_PD difference (≈0.003 µm), so do not read run-to-run differences of the global in-cloud lines.


In [ ]:
import re
RUNS = {
    "PD":  ["CTRL_PD", "LCC_PD", "LCC_PD_fBVOCctrlPD", "LCC_PD_fBVOClccPD", "CTRL_PD_fBVOCctrlPD"],
    "FUT": ["CTRL_FUT", "LCC_FUT", "LCC_FUT_fBVOCctrlPD", "LCC_FUT_fBVOCctrlFUT", "LCC_FUT_fBVOClccPD",
            "CTRL_FUT_fBVOCctrlFUT"],
}
def run_style(alias):
    if alias.startswith("CTRL"):
        return pal.NAVY, ("--" if "fBVOC" in alias else "-")
    return {"ctrlPD": pal.PLUM, "ctrlFUT": pal.ORANGE, "lccPD": pal.TEAL}.get(alias.split("fBVOC")[-1], pal.RED), "-"

def run_span(alias, cls, comp):
    """(y0, y1) of the single processed file of this run and class."""
    files = sorted(fm.POSTPROCESSED_DIR.glob(f"{alias}_{comp}_{cls}_*-*.nc"))
    assert len(files) == 1, (alias, cls, [f.name for f in files])
    return tuple(int(x) for x in re.search(r"_(\d{4})-(\d{4})\.nc$", files[0].name).groups())

rows, opened = [], {}
for period, aliases in RUNS.items():
    for alias in aliases:
        for key, title, cls, var, units, comp in PANELS:
            if (alias, cls, comp) not in opened:
                y0, y1 = run_span(alias, cls, comp)
                opened[alias, cls, comp] = fm.open_class(alias, cls, y0, y1, comp)
            ds = opened[alias, cls, comp]
            f = fm.yearly_field(ds, var)
            assert np.allclose(ds["lat"], refs[comp]["lat"]) and np.allclose(ds["lon"], refs[comp]["lon"])
            f = f.assign_coords(lat=refs[comp]["lat"], lon=refs[comp]["lon"])   # CAM/CLM coords differ by ~1e-13
            for line, region in (("global", "global"), ("regional", REGIONAL.get(key, "≥45°N"))):
                r = fm.regional(f, refs[comp], masks[comp][region])
                rows += [dict(period=period, run=alias, panel=key, variable=var, line=line, region=region,
                              units=units, year=int(y), value=float(v)) for y, v in zip(r["year"].values, r.values)]
a = pd.DataFrame(rows)

g = a[a.panel.str.startswith("NET_")].pivot_table(index=["run", "line", "year"], columns="variable", values="value")
err = (g["NET_dir"] + g["NET_cloud"] + g["NET_rest"] - g["NET_toa"]).abs().max()
print(f"Ghan closure (absolute regional means): {err:.1e} W m-2"); assert err < 1e-3
print(a.groupby("run").year.agg(["min", "max"]).drop_duplicates())

ABS_TABLE = TABLE.with_name(f"timeseries_absolute_{START_YEAR}_{END_YEAR}.csv")
a.to_csv(ABS_TABLE, index=False); print("Saved", ABS_TABLE)

# consistency: absolute differences reproduce section 1 (Total_PD, ≥45°N / modified cells)
chk = a[a.run.isin(["LCC_PD", "CTRL_PD"])].pivot_table(index=["panel", "line", "year"], columns="run", values="value")
ref = t[t.contrast == "LCC-CTRL_PD"].set_index(["panel", "line", "year"]).value
err = ((chk.LCC_PD - chk.CTRL_PD) - ref).abs().groupby(["panel", "line"]).max()
print("max |(LCC_PD - CTRL_PD) - section-1 difference| per panel:")
print(pd.DataFrame({"max err": err, "mean |diff|": ref.abs().groupby(["panel", "line"]).mean()}).sort_values("max err", ascending=False).head(6))
# exact for all fields except the in-cloud ones: their FCTL >= 0.01 mask differs between runs
assert err.drop(["actnl", "actrel"], level="panel").max() < 1e-6


In [ ]:
# mean over years per run (regional line), for reading the levels
with pd.option_context("display.max_rows", 200, "display.width", 200):
    display(a[a.line == "regional"].groupby(["period", "panel", "run"], sort=False).value.mean()
            .unstack("run").round(4))


In [ ]:
def plot_absolute(period, line, save=True):
    sub = a[(a.period == period) & (a.line == line)]
    fig = plt.figure(figsize=(13, 14), constrained_layout=True)
    top, bottom = fig.subfigures(2, 1, height_ratios=[3, 1.05], hspace=0.04)
    axes = list(top.subplots(3, 3).ravel()) + list(bottom.subplots(1, 4))
    bottom.suptitle("Ghan components, NET, absolute (down positive)", fontsize=11, fontweight="semibold")
    for ax, (key, title, cls, var, units, comp) in zip(axes, PANELS):
        d = sub[sub.panel == key]
        for alias in RUNS[period]:
            r = d[d.run == alias].sort_values("year")
            color, ls = run_style(alias)
            ax.plot(r.year, r.value, marker="o", ms=3, lw=1.3, ls=ls, color=color, label=alias)
        region = d.region.iloc[0]
        ax.set_title(f"{title}\n({region})", fontsize=9.5, fontweight="semibold")
        ax.set_ylabel(f"[{units}]", fontsize=8)
        ax.set_xticks(range(START_YEAR, END_YEAR + 1, 2)); ax.tick_params(labelsize=7.5)
        ax.grid(alpha=0.25, lw=0.5)
    h, l = axes[0].get_legend_handles_labels()
    fig.legend(h, l, loc="outside lower center", ncol=3, fontsize=8.5, frameon=False)
    where = "global mean" if line == "global" else "modified cells (BVOC emissions) / ≥45°N (other variables)"
    fig.suptitle(f"Absolute yearly means, {period} runs ({START_YEAR}–{END_YEAR}): {where}",
                 fontsize=12, fontweight="bold")
    if save:
        path = FIGDIR / f"timeseries_absolute_{period}_{line}_{START_YEAR}_{END_YEAR}.png"
        fig.savefig(path, dpi=200, bbox_inches="tight"); print("Saved", path)
    return fig

for period in RUNS:
    for line in ("global", "regional"):
        fig = plot_absolute(period, line)
        plt.show(); plt.close(fig)
